## Environment info

Distro: Linux Fedora 43 for Workstation </br>
PyTorch: 2.11.0 (Stable)</br></br>
Calculations are being done on an Intel i5-12600K with an x86 instruction set. </br></br>
Running in `conda` environment with Python 3.13.3


Currently running kernel:

In [1]:
!cat /proc/version

Linux version 6.19.13-200.fc43.x86_64 (mockbuild@f1d307f8ec4b43aabb80cc7c7a34b562) (gcc (GCC) 15.2.1 20260123 (Red Hat 15.2.1-7), GNU ld version 2.45.1-4.fc43) #1 SMP PREEMPT_DYNAMIC Sat Apr 18 20:20:44 UTC 2026


List of packages and versions in environment: </br>
> <strong>Note:</strong> This environment contains a lot of NVIDIA CUDA packages (CUDA v12.8) because I installed `torch` with CUDA. </br>
> NVIDIA CUDA will most likely not run on MacOS but is not needed for this example. If you are on MacOS and have trouble getting `torch` to work, try to specifically install the cpu-only version without CUDA.

In [2]:
!conda list -n ml_environment_cuda128

# packages in environment at /home/simon/anaconda3/envs/ml_environment_cuda128:
#
# Name                       Version          Build                  Channel
_libgcc_mutex                0.1              main
_openmp_mutex                5.1              1_gnu
anyio                        4.12.1           py313h06a4308_0
argon2-cffi                  25.1.0           py313h06a4308_0
argon2-cffi-bindings         25.1.0           py313hee96239_0
asttokens                    3.0.1            py313h06a4308_0
async-lru                    2.0.5            py313h06a4308_0
attrs                        26.1.0           py313h0c820a0_0
babel                        2.17.0           py313h06a4308_0
beautifulsoup4               4.14.3           py313h06a4308_0
blas                         1.0              mkl
bleach                       6.3.0            py313h06a4308_0
brotlicffi                   1.2.0.0          py313h7354ed3_0
bzip2                        1.0.8            h5eee18b_6
ca-certific

</br></br></br>
Checking python install. </br>
Both of the commands below should point to the same location otherwise some problems might come up. </br>
><strong>Note:</strong> `!which python` is for Linux, on Windows you would probably type `!where python` but you can try both.

In [3]:
import sys
print(sys.executable)

/home/simon/anaconda3/envs/ml_environment_cuda128/bin/python


In [4]:
!which python

/home/simon/anaconda3/envs/ml_environment_cuda128/bin/python


</br></br></br></br></br>

## Data preparations

In [5]:
import pandas as pd

In [6]:
#import dataset
dataset = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-15/spam.csv")

#verify
dataset.info()
dataset.head()

<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crl.tot  4601 non-null   int64  
 1   dollar   4601 non-null   float64
 2   bang     4601 non-null   float64
 3   money    4601 non-null   float64
 4   n000     4601 non-null   float64
 5   make     4601 non-null   float64
 6   yesno    4601 non-null   str    
dtypes: float64(5), int64(1), str(1)
memory usage: 251.7 KB


,crl.tot,dollar,bang,money,n000,make,yesno
0,278,0.000,0.778,0.00,0.00,0.00,y
1,1028,0.180,0.372,0.43,0.43,0.21,y
2,2259,0.184,0.276,0.06,1.16,0.06,y
3,191,0.000,0.137,0.00,0.00,0.00,y
4,191,0.000,0.135,0.00,0.00,0.00,y


In [7]:
#get some info about the dataset
dataset.describe()

,crl.tot,dollar,bang,money,n000,make
count,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000
mean,283.289285,0.075811,0.269071,0.094269,0.101645,0.104553
std,606.347851,0.245882,0.815672,0.442636,0.350286,0.305358
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,35.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,95.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,266.000000,0.052000,0.315000,0.000000,0.000000,0.000000
max,15841.000000,6.003000,32.478000,12.500000,5.450000,4.540000


In [8]:
#recode values for yesno
dataset["yesno"] = dataset["yesno"].replace({
    "n": "not_spam",
    "y": "spam"
}
                                           )

#verify
dataset.head()

,crl.tot,dollar,bang,money,n000,make,yesno
0,278,0.000,0.778,0.00,0.00,0.00,spam
1,1028,0.180,0.372,0.43,0.43,0.21,spam
2,2259,0.184,0.276,0.06,1.16,0.06,spam
3,191,0.000,0.137,0.00,0.00,0.00,spam
4,191,0.000,0.135,0.00,0.00,0.00,spam


In [9]:
#convert to categorical
dataset["yesno"] = pd.Categorical(
    dataset["yesno"],
    categories=["not_spam", "spam"],
    ordered=True
)


In [10]:
from sklearn.model_selection import train_test_split, StratifiedKFold

#define test and training data
train_data, test_data = train_test_split(
    dataset,
    test_size = 0.8, #set test dataset size
    stratify = dataset["yesno"], #stratify
    random_state = 42 #set seed
)

In [11]:
#define cross-validation
validation_folds = StratifiedKFold(n_splits = 5, #default, but lets specify it anyway
                      shuffle = True, 
                      random_state = 42 #set seed
                                  )

#capitalize X but not y by convention
X = train_data.drop(columns=["yesno"])
y = train_data["yesno"]

folds = list(validation_folds.split(X, y))

#print first fold sizes
train_idx, val_idx = folds[0]
print(len(train_idx), len(val_idx))

736 184


</br></br></br></br></br>

## Random forest model

In [12]:
#get functions
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [13]:
#separate features and target
X_train = train_data.drop(columns=["yesno"])
y_train = train_data["yesno"]

X_test = test_data.drop(columns=["yesno"])
y_test = test_data["yesno"]

In [14]:
#specify random forest model
random_forest_model = RandomForestClassifier(
    
    #specify number of trees
    n_estimators = 500,

    #let trees grow indefinietely
    max_depth = None,
    
    min_samples_split = 2,

    #set seed
    random_state = 42,

    #use all available cpu cores
    n_jobs = -1
)

In [15]:
#train model
random_forest_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

Conduct cross-validation. The data is split into 5 chunks. Four of them are used for training and one for testing. This is repeated five times, each time with a different chunk for testing. </br>
`scores` gives the accuracy for each run. 

In [16]:
from sklearn.model_selection import cross_val_score

#conduct cross-validation
scores = cross_val_score(
    
    #model
    random_forest_model,

    #input data
    X_train,

    #truth
    y_train,

    #number of cross-validations
    cv = 5,

    #performance metric
    scoring = "accuracy"
)

print("cross-validation accuracy", scores)
print("cross-validation accuracy (mean):", scores.mean())

cross-validation accuracy [0.88586957 0.90217391 0.875      0.83152174 0.83695652]
cross-validation accuracy (mean): 0.8663043478260871


Use the test data (= X_test) to make predictions for evaluation.

In [17]:
#predict
y_predict = random_forest_model.predict(X_test)

Compute the confusion matrix to check true_positives etc.

In [18]:
from sklearn.metrics import confusion_matrix

#compute confusion matrix
cm = confusion_matrix(y_test, y_predict)


#add labels
cm_df = pd.DataFrame(
    cm,
    index=["True: not_spam", "True: spam"],
    columns=["Estimate: not_spam", "Estimate: spam"]
)

print(cm_df)

                Estimate: not_spam  Estimate: spam
True: not_spam                2024             207
True: spam                     316            1134


Finally, get performance metrics by comparing predictions (= y_predict) against the truth (=y_test).

In [19]:
#get performance metrics
accuracy_random_forest = accuracy_score(y_test, y_predict)
print("accuracy:", accuracy_random_forest)

print("performance metrics: \n\n", #start new line with \n to keep headers where they belong
      classification_report(y_test, y_predict))

accuracy: 0.8579190437381147
performance metrics: 

               precision    recall  f1-score   support

    not_spam       0.86      0.91      0.89      2231
        spam       0.85      0.78      0.81      1450

    accuracy                           0.86      3681
   macro avg       0.86      0.84      0.85      3681
weighted avg       0.86      0.86      0.86      3681



</br></br></br></br></br>

## Pytorch neural network

For this kind of data, a neural net might not necessarily make a lot of sense and also bring no benefit comapred to random forest. But lets run it anyway.

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler

In [21]:
#convert target to integer
y_train = (y_train == "spam").astype(int)
y_test = (y_test == "spam").astype(int)

Scale the features so they are all roughly the same size. Scikit's `StandardScaler()` scales like this: </br></br>
$z = \dfrac{x - M}{\sigma^2}$ </br></br>
with: </br>
$z$ = scaled value </br>
$x$ = unscaled value </br>
$M$ = mean of all values in that category </br>
$\sigma^2$ = standard deviation of all values in that category </br></br>
Because of the usage of mean and standard deviation `StandardScaler()` is sensitive to outliers but lets ignore that for this example.

In [22]:
#scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Currently the data is ordered as a `numpy` array:

In [23]:
type(X_train)

numpy.ndarray

However, Pytorch requires tensors. So convert the data to tensors:

In [24]:
#convert data to tensors
X_train_tensor = torch.tensor(X_train, 
                       dtype = torch.float32
                      )

y_train_tensor = torch.tensor(y_train.values, 
                       dtype = torch.float32
                      ).view(-1, 1)

X_test_tensor = torch.tensor(X_test, 
                      dtype = torch.float32
                     )

y_test_tensor = torch.tensor(y_test.values, 
                      dtype = torch.float32
                     ).view(-1, 1)

print(type(X_train_tensor))

<class 'torch.Tensor'>


Next, define how many input dimensions there are:

In [25]:
input_dimensions = X_train.shape[1]
print(input_dimensions)

6


Now define a pipeline for the model: </br></br>
`nn.Module` is the base for neural nets in Pytorch. </br>
Next a setup function (so-called "constructor") is defined. `input_dimensions` will tell the model how many dimensions exist. </br>
`super(ModelClass, self).__init__()` calls on the parent class to initialize the built-in neural network functionalities. </br></br>
Afterwards, the layers of the model are added: </br>

- <strong>nn.Sequential():</strong> Runs commands in a specified order.
- <strong>nn.Linear():</strong> Takes input numbers, multiplies them with weights and adds a bias. Basically a weighted sum. Called "linear" because it only consists of linear math.
- <strong>nn.ReLU():</strong> Adds non-linearity to the calculation. Applies rules to each number. Basically a filter.
- <strong>nn.Sigmoid():</strong> Converts all numbers to a value between 0 and 1. Similar to probabilities.

In [26]:
#define model

#sub-class nn.Module
class ModelClass(nn.Module):

    #constructor
    def __init__(self, 
                 input_dimensions):

        #initialize pytorch
        super(ModelClass, 
              self).__init__()

        #define layers
        self.model = nn.Sequential(
            nn.Linear(input_dimensions, 16), #16 neurons
            nn.ReLU(), #activation function
            nn.Linear(16, 8), #8 neurons
            nn.ReLU(), #activation function
            nn.Linear(8, 1), #1 output
            
            nn.Sigmoid() #apply sigmoid function
        )

    #define forward passing
    def forward(self, x):
        return self.model(x)

In [27]:
model = ModelClass(input_dimensions)

Next set a criterion for measuring error. The model will make a prediction between 0 and 1. In this case the real answer (=truth) is always either 1 or 0. Binary Cross Entropy is used as criterion and will compare how close the prediction is to the truth. </br>
Optimizer defines how the model will improve itself based on the measured error by the criterion. `Adam` is a method for optimization (Adam = Adaptive Moment Estimation). `lr` defines the learning rate (i.e. how big the adjustment steps are).

In [28]:
#use binary cross entropy as criterion
criterion = nn.BCELoss()

#use adam as method for optimizer
optimizer = optim.Adam(model.parameters(), lr = 0.001)

Now the training itself. </br>
Trainig is conducted for 250 epochs. This means the model will go over th training data 250 times. </br>
The model will look at the input data (= X_train_tensor) and produce predictions (= outputs). </br>
Afterwards, the model will compare the predictions with the correct answers (= y_train_tensor). </br>
Then it can use a lot of math to try and find out what caused the errors (= loss.backwards()) and how to fix them (= optimizer.step()). </br>

In [29]:
#define training loop

#train for 250 epochs
epochs = 250

for epoch in range(epochs):
    
    #tell model it is being trained
    model.train()

    #clear gradients to avoid accumulation
    optimizer.zero_grad()

    #make predictions
    outputs = model(X_train_tensor)
    
    #compare predictions to truth
    loss = criterion(outputs, y_train_tensor)

    #calculate why errors happened
    loss.backward()

    #adjust according to optimizer
    optimizer.step()

    #define print() for updates after amount of epochs
    if (epoch + 1) % 10 == 0: #the % conducts modulo-operations
        
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.6862
Epoch 20, Loss: 0.6776
Epoch 30, Loss: 0.6687
Epoch 40, Loss: 0.6594
Epoch 50, Loss: 0.6485
Epoch 60, Loss: 0.6359
Epoch 70, Loss: 0.6218
Epoch 80, Loss: 0.6062
Epoch 90, Loss: 0.5888
Epoch 100, Loss: 0.5703
Epoch 110, Loss: 0.5514
Epoch 120, Loss: 0.5331
Epoch 130, Loss: 0.5145
Epoch 140, Loss: 0.4957
Epoch 150, Loss: 0.4776
Epoch 160, Loss: 0.4605
Epoch 170, Loss: 0.4445
Epoch 180, Loss: 0.4300
Epoch 190, Loss: 0.4168
Epoch 200, Loss: 0.4050
Epoch 210, Loss: 0.3944
Epoch 220, Loss: 0.3852
Epoch 230, Loss: 0.3772
Epoch 240, Loss: 0.3703
Epoch 250, Loss: 0.3643


Finally, evaluate performance on the test data (= X_test_tensor) and compare the predictions against the truth (= y_test_tensor).

In [30]:
#evaluate performance

#tell model it is being evaluated
model.eval()

#disable all learning calculations for efficiency
with torch.no_grad():

    #make predictions on test data
    predictions = model(X_test_tensor)

    #dichotomize to 0 and 1
    predictions_dich = (predictions > 0.5).float()

#compare predictions against truth
accuracy_pytorch_neuralnet = (predictions_dich == y_test_tensor).float().mean()
print("Test Accuracy:", accuracy_pytorch_neuralnet.item())

Test Accuracy: 0.8478674292564392


In [31]:
#compare to random forest
print("Accuracy Pytorch neural network:", accuracy_pytorch_neuralnet.item())
print("Accuracy random forest model:", accuracy_random_forest)
print("Delta:", accuracy_pytorch_neuralnet.item() - accuracy_random_forest)

Accuracy Pytorch neural network: 0.8478674292564392
Accuracy random forest model: 0.8579190437381147
Delta: -0.010051614481675464
